Define the input paths

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime

In [0]:
IND_PATH = "/Volumes/workspace/skypoints/raw_files/IND.csv"
USA_PATH = "/Volumes/workspace/skypoints/raw_files/USA.csv"
AUS_PATH = "/Volumes/workspace/skypoints/raw_files/AUS.csv"

BATCH_ID = "BATCH_20261121_001"

Create the Bronze schema and table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS
workspace.skypoints.bronze_member_feed
(
    RAW_RECORD STRING,
    SOURCE_COUNTRY STRING,
    SOURCE_FILE_NAME STRING,
    INGESTION_TIMESTAMP TIMESTAMP,
    BATCH_ID STRING,
    RECORD_HASH STRING
)
USING DELTA;

In [0]:
ind_df = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .load(IND_PATH)
)

usa_df = (
    spark.read
    .option("header",True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(USA_PATH)
)

aus_df = (
    spark.read
    .option("header",True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(AUS_PATH)
)

Convert source records into Bronze format

In [0]:
def prepare_bronze_data(df, country, file_name):

    raw_columns = [F.col(f"`{column}`") for column in df.columns]

    bronze_df = df.select(
        F.to_json(F.struct(*raw_columns)).alias("RAW_RECORD"),
        F.lit(country).alias("SOURCE_COUNTRY"),
        F.lit(file_name).alias("SOURCE_FILE_NAME"),
        F.current_timestamp().alias("INGESTION_TIMESTAMP"),
        F.lit(BATCH_ID).alias("BATCH_ID")
    )

    bronze_df = bronze_df.withColumn(
        "RECORD_HASH", F.sha2(F.col("RAW_RECORD"), 256)
    )

    return bronze_df



In [0]:
ind_bronze = prepare_bronze_data(ind_df, "IND","IND.CSV")
usa_bronze = prepare_bronze_data(usa_df, "USA", "USA.csv")
aus_bronze = prepare_bronze_data(aus_df, "AUS", "AUS.csv")

In [0]:
bronze_df = (
    ind_bronze
    .unionByName(usa_bronze)
    .unionByName(aus_bronze)
)

In [0]:
display(bronze_df)

In [0]:
target_table = "workspace.skypoints.bronze_member_feed"


(
    bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(target_table)
)

Make ingestion idempotent

In [0]:
existing_batch = (
    spark.table(target_table)
    .filter(F.col("BATCH_ID") == BATCH_ID)
    .limit(1)
    .count()
)

if existing_batch > 0:
    print(f"Batch {BATCH_ID} already loaded. Skipping ingestion.")
else:
    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(target_table)
    )
    print(f"Batch {BATCH_ID} loaded successfully.")

Validate the Bronze data

In [0]:
%sql
SELECT COUNT(*) AS TOTAL_RECORDS
FROM skypoints.bronze_member_feed;

In [0]:
%sql
SELECT
    SOURCE_COUNTRY,
    SOURCE_FILE_NAME,
    COUNT(*) AS RECORD_COUNT
FROM skypoints.bronze_member_feed
GROUP BY
    SOURCE_COUNTRY,
    SOURCE_FILE_NAME
ORDER BY SOURCE_COUNTRY;

In [0]:
%sql
SELECT
    SOURCE_COUNTRY,
    SOURCE_FILE_NAME,
    RAW_RECORD,
    BATCH_ID,
    INGESTION_TIMESTAMP
FROM skypoints.bronze_member_feed;

In [0]:
%sql
--Check for duplicate records:
SELECT
    SOURCE_COUNTRY,
    RECORD_HASH,
    COUNT(*) AS DUPLICATE_COUNT
FROM skypoints.bronze_member_feed
GROUP BY
    SOURCE_COUNTRY,
    RECORD_HASH
HAVING COUNT(*) > 1;